# 🩺 Diabetes Prediction using Machine Learning
**Self-Evaluation Project**

**Objective:** Apply everything learned so far to build a complete machine learning classification project using the Pima Indians Diabetes dataset, with special focus on real-world data issues — specifically, zero-encoded missing values.

**Dataset:** `diabetes.csv`  
**Kaggle Link:** https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

---
**Target:**
- `0` = No Diabetes
- `1` = Diabetes (Positive)


## 1. Setup

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings("ignore")

# ── Configuration ──────────────────────────────────────────────────────────
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
sns.set_theme(style="darkgrid")

plt.rcParams.update({
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
})

RANDOM_STATE = 42
CSV_PATH     = "diabetes.csv"
TARGET_COL   = "Outcome"

# Columns where 0 is biologically impossible → encoded missing values
ZERO_MISSING_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]


## 2. Load Data

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()

In [ ]:
df.dtypes

## 3. Exploratory Data Analysis (EDA)

### 3.1 Basic Overview

In [ ]:
# Missing values check (standard nulls)
print("Standard missing values (NaN):")
print(df.isnull().sum())


In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
# Descriptive statistics
df.describe().T


### 3.2 Target Distribution

In [ ]:
print("Target value counts:")
print(df[TARGET_COL].value_counts())
print()
print("Proportion (%):" )
print(df[TARGET_COL].value_counts(normalize=True) * 100)


In [ ]:
plt.figure(figsize=(5, 3))
sns.countplot(x=TARGET_COL, data=df, palette="Set2")
plt.title("Target Distribution — Diabetes Outcome")
plt.xticks([0, 1], ["No Diabetes (0)", "Diabetes (1)"])
plt.ylabel("Count")
plt.tight_layout()
plt.show()


### 3.3 Zero-Encoded Missing Values

> ⚠️ **Critical Preprocessing Note:**  
> In this dataset, certain medical features contain `0` values that are **biologically impossible** — a person cannot have a Blood Pressure or BMI of zero.  
> These zeros are **not valid measurements** — they represent missing data that was encoded as `0`.  
> They must be identified, replaced with `NaN`, and then imputed before modelling.


In [ ]:
print("Zero counts per column (before replacement):")
print("-" * 50)
for col in df.columns:
    zeros = (df[col] == 0).sum()
    pct   = zeros / len(df) * 100
    flag  = " ⚠️  ENCODED MISSING" if col in ZERO_MISSING_COLS and zeros > 0 else ""
    print(f"  {col:<28} {zeros:>4} zeros  ({pct:>5.1f}%){flag}")


In [ ]:
# Visualise zero distribution across medical features
fig, axes = plt.subplots(1, len(ZERO_MISSING_COLS), figsize=(14, 3))

for ax, col in zip(axes, ZERO_MISSING_COLS):
    zero_count   = (df[col] == 0).sum()
    nonzero_count = len(df) - zero_count
    ax.bar(["Non-zero", "Zero (Missing)"], [nonzero_count, zero_count],
           color=["#4CAF50", "#F44336"])
    ax.set_title(col, fontsize=9)
    ax.set_ylabel("Count")

plt.suptitle("Zero-Encoded Missing Values per Feature", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


### 3.4 Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()
feature_cols = [c for c in df.columns if c != TARGET_COL]

for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=30, color="steelblue", edgecolor="white")
    axes[i].set_title(col)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Count")

plt.suptitle("Feature Distributions", fontsize=12)
plt.tight_layout()
plt.show()


### 3.5 Outlier Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor="lightblue"))
    axes[i].set_title(col)

plt.suptitle("Outlier Analysis — Boxplots", fontsize=12)
plt.tight_layout()
plt.show()


### 3.6 Correlation Heatmap

In [ ]:
plt.figure(figsize=(9, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            mask=mask, linewidths=0.5, square=True)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


### 3.7 Feature Distributions by Target

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    for outcome, colour in zip([0, 1], ["#4CAF50", "#F44336"]):
        axes[i].hist(df[df[TARGET_COL] == outcome][col].dropna(),
                     bins=25, alpha=0.6, color=colour,
                     label="No Diabetes" if outcome == 0 else "Diabetes")
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)

plt.suptitle("Feature Distributions by Outcome", fontsize=12)
plt.tight_layout()
plt.show()


## 4. Data Preprocessing

### 4.1 Replace Zero-Encoded Missing Values with NaN


In [ ]:
# Replace biologically impossible zeros with NaN
df[ZERO_MISSING_COLS] = df[ZERO_MISSING_COLS].replace(0, np.nan)

print("Missing values after replacement:")
print(df.isnull().sum())
print()
print("Total missing:", df.isnull().sum().sum())


### 4.2 Features and Target Split

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print("Features shape:", X.shape)
print("Target shape:  ", y.shape)
print("Feature columns:", X.columns.tolist())


### 4.3 Train/Test Split (Stratified 80/20)

`stratify=y` preserves the class ratio (65% No Diabetes / 35% Diabetes) in both training and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    stratify     = y,
    random_state = RANDOM_STATE
)

print(f"Training set   : {X_train.shape[0]} rows")
print(f"Test set       : {X_test.shape[0]} rows")
print()
print("Train target distribution:")
print(y_train.value_counts(normalize=True).rename({0:"No Diabetes",1:"Diabetes"}) * 100)
print()
print("Test target distribution:")
print(y_test.value_counts(normalize=True).rename({0:"No Diabetes",1:"Diabetes"}) * 100)


### 4.4 Preprocessing Pipeline

Two steps chained in a `Pipeline`:
1. **`SimpleImputer(strategy='median')`** — fills NaN with the median of each feature (computed on training data only, to prevent data leakage)
2. **`StandardScaler()`** — scales each feature to zero mean and unit variance

> **Why median imputation?**  
> Median is more robust to outliers than mean — important here because Insulin and SkinThickness have extreme zero clusters that, once replaced, leave many NaNs. The median gives a representative central value without being pulled by extremes.


In [ ]:
preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])


## 5. Model Training & Selection

### 5.1 Baseline — Logistic Regression


In [ ]:
baseline_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model",      LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
])

baseline_pipe.fit(X_train, y_train)
y_pred_baseline = baseline_pipe.predict(X_test)

print("=== Baseline — Logistic Regression (Test) ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_baseline):.4f}")
print(f"Recall   : {recall_score(y_test,   y_pred_baseline):.4f}")
print(f"F1-Score : {f1_score(y_test,        y_pred_baseline):.4f}")
print()
print(classification_report(y_test, y_pred_baseline,
                             target_names=["No Diabetes", "Diabetes"]))


### 5.2 Bonus — Model Comparison with 5-Fold Stratified Cross-Validation

Four models are compared using `StratifiedKFold(n_splits=5)`.  
**Primary metric: F1-Score** — balances Precision and Recall for the imbalanced target.  
**Secondary metric: Recall** — ensures we do not miss diabetic patients (false negatives are costly).


In [ ]:
models = {
    "LogisticRegression" : LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    "RandomForest"       : RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200),
    "GradientBoosting"   : GradientBoostingClassifier(random_state=RANDOM_STATE),
    "SVC (RBF)"          : SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
print(f"{'Model':<22} | {'CV Accuracy':>11} | {'CV Recall':>9} | {'CV F1':>7}")
print("-" * 58)

for name, clf in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", clf)])
    cv   = cross_validate(
        pipe, X_train, y_train,
        cv      = skf,
        scoring = ["accuracy", "recall", "f1"],
        n_jobs  = -1
    )
    cv_results[name] = {
        "acc"   : cv["test_accuracy"].mean(),
        "recall": cv["test_recall"].mean(),
        "f1"    : cv["test_f1"].mean(),
    }
    print(f"{name:<22} | {cv['test_accuracy'].mean():>11.4f} | "
          f"{cv['test_recall'].mean():>9.4f} | {cv['test_f1'].mean():>7.4f}")


In [ ]:
# Visualise CV results
cv_df = pd.DataFrame(cv_results).T.reset_index().rename(columns={"index":"Model"})

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = [("acc", "CV Accuracy"), ("recall", "CV Recall"), ("f1", "CV F1-Score")]

for ax, (col, title) in zip(axes, metrics):
    bars = ax.barh(cv_df["Model"], cv_df[col],
                   color=sns.color_palette("Set2", len(cv_df)))
    ax.set_title(title)
    ax.set_xlim(0.5, 0.85)
    ax.axvline(cv_df[col].max(), color="red", linestyle="--", linewidth=0.8, alpha=0.7)
    for bar, val in zip(bars, cv_df[col]):
        ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=8)

plt.suptitle("5-Fold Cross-Validation Comparison", fontsize=12)
plt.tight_layout()
plt.show()


### 5.3 Select Best Model & Train on Full Training Set

In [ ]:
# Best model by F1-Score
best_name = max(cv_results, key=lambda k: cv_results[k]["f1"])
print(f"Best model by CV F1-Score: {best_name}")
print(f"  CV F1    : {cv_results[best_name]['f1']:.4f}")
print(f"  CV Recall: {cv_results[best_name]['recall']:.4f}")
print(f"  CV Acc   : {cv_results[best_name]['acc']:.4f}")


In [ ]:
# Retrain best model on full training set
best_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model",      models[best_name])
])
best_pipe.fit(X_train, y_train)
print(f"{best_name} retrained on all {X_train.shape[0]} training rows.")


## 6. Model Evaluation

### 6.1 Full Model Comparison on Test Set


In [ ]:
print(f"{'Model':<22} | {'Test Acc':>8} | {'Test Recall':>11} | {'Test F1':>8}")
print("-" * 60)

test_results = {}
for name, clf in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", clf)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_test)
    test_results[name] = {
        "acc"   : accuracy_score(y_test, yp),
        "recall": recall_score(y_test, yp),
        "f1"    : f1_score(y_test, yp),
    }
    print(f"{name:<22} | {accuracy_score(y_test,yp):>8.4f} | "
          f"{recall_score(y_test,yp):>11.4f} | {f1_score(y_test,yp):>8.4f}")


### 6.2 Best Model — Detailed Test Report

In [ ]:
y_pred = best_pipe.predict(X_test)

print(f"=== {best_name} — Test Set Results ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred,
                             target_names=["No Diabetes", "Diabetes"],
                             digits=4))


### 6.3 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Train confusion matrix
y_pred_train = best_pipe.predict(X_train)
ConfusionMatrixDisplay(
    confusion_matrix(y_train, y_pred_train),
    display_labels=["No Diabetes", "Diabetes"]
).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"{best_name} — Train")

# Test confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=["No Diabetes", "Diabetes"]
).plot(ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title(f"{best_name} — Test")

plt.suptitle("Confusion Matrices", fontsize=12)
plt.tight_layout()
plt.show()


### 6.4 Understanding the Confusion Matrix

| | Predicted No Diabetes | Predicted Diabetes |
|---|---|---|
| **Actual No Diabetes** | ✅ True Negative (TN) | ❌ False Positive (FP) |
| **Actual Diabetes** | ⚠️ False Negative (FN) | ✅ True Positive (TP) |

> **Why False Negatives matter most here:**  
> A False Negative means a diabetic patient is predicted as healthy — they receive no treatment.  
> In medical screening, this is the most dangerous error. This is why **Recall** is tracked alongside Accuracy.


## 7. Predictive System

In [ ]:
def predict_diabetes(model, pregnancies, glucose, blood_pressure,
                      skin_thickness, insulin, bmi,
                      diabetes_pedigree, age):
    """
    Predict diabetes outcome for a new patient.

    Parameters
    ----------
    model            : trained sklearn pipeline
    pregnancies      : number of pregnancies
    glucose          : plasma glucose concentration (mg/dL)
    blood_pressure   : diastolic blood pressure (mm Hg) — use np.nan if unknown
    skin_thickness   : triceps skin fold thickness (mm) — use np.nan if unknown
    insulin          : 2-hour serum insulin (mu U/ml) — use np.nan if unknown
    bmi              : body mass index (weight/height²) — use np.nan if unknown
    diabetes_pedigree: diabetes pedigree function score
    age              : age in years

    Returns
    -------
    Prints prediction label and confidence score.
    """
    input_data = pd.DataFrame([{
        "Pregnancies"             : pregnancies,
        "Glucose"                 : glucose,
        "BloodPressure"           : blood_pressure,
        "SkinThickness"           : skin_thickness,
        "Insulin"                 : insulin,
        "BMI"                     : bmi,
        "DiabetesPedigreeFunction": diabetes_pedigree,
        "Age"                     : age
    }])

    prediction   = model.predict(input_data)[0]
    confidence   = model.predict_proba(input_data)[0][prediction] * 100

    label = "Diabetes Positive 🔴" if prediction == 1 else "No Diabetes 🟢"
    print(f"Prediction  : {label}")
    print(f"Confidence  : {confidence:.1f}%")
    print(f"Raw output  : {prediction}")
    return prediction


In [ ]:
# ── Example 1: High-risk patient ──────────────────────────────────────────
print("=== Patient 1 — High Risk ===")
predict_diabetes(
    model             = best_pipe,
    pregnancies       = 6,
    glucose           = 148,
    blood_pressure    = 72,
    skin_thickness    = 35,
    insulin           = np.nan,   # unknown — pipeline imputes
    bmi               = 33.6,
    diabetes_pedigree = 0.627,
    age               = 50
)


In [ ]:
# ── Example 2: Low-risk patient ───────────────────────────────────────────
print("=== Patient 2 — Low Risk ===")
predict_diabetes(
    model             = best_pipe,
    pregnancies       = 1,
    glucose           = 85,
    blood_pressure    = 66,
    skin_thickness    = 29,
    insulin           = 0,        # zero = unknown → will be treated as NaN by imputer
    bmi               = 26.6,
    diabetes_pedigree = 0.351,
    age               = 31
)


In [ ]:
# ── Example 3: New unseen patient ─────────────────────────────────────────
print("=== Patient 3 — New Unseen Patient ===")
predict_diabetes(
    model             = best_pipe,
    pregnancies       = 3,
    glucose           = 130,
    blood_pressure    = 78,
    skin_thickness    = np.nan,   # unknown
    insulin           = np.nan,   # unknown
    bmi               = 31.2,
    diabetes_pedigree = 0.490,
    age               = 45
)


---

## 8. Summary

| Step | What Was Done |
|---|---|
| **Data Loading** | Loaded 768 rows × 9 columns from `diabetes.csv` |
| **EDA** | Confirmed class imbalance (65% / 35%), identified zero-encoded missings |
| **Zero-Encoded Missing Values** | Replaced 0s in 5 columns with NaN — up to 48.7% missing in Insulin |
| **Imputation** | `SimpleImputer(strategy='median')` — fit on training data only |
| **Scaling** | `StandardScaler()` applied inside Pipeline to prevent data leakage |
| **Train/Test Split** | 80/20 stratified split — 614 train, 154 test |
| **Baseline** | Logistic Regression — Accuracy 70.8%, Recall 50.0%, F1 54.6% |
| **Model Comparison** | 4 models compared via 5-fold stratified CV, scored on F1 and Recall |
| **Evaluation** | Full classification report + confusion matrices (train and test) |
| **Predictive System** | `predict_diabetes()` — handles unknown values via np.nan |

### Key Learning from this Project
> Zero-encoded missing values are a **real-world data quality issue** — not a textbook problem.  
> Treating zeros in `Insulin` and `SkinThickness` as valid measurements would have given the model corrupted information, directly harming prediction quality.  
> Correct handling — replace with NaN, then median impute inside a pipeline — is what separates professional ML practice from beginner mistakes.
